# SIT307 HD evidence verification

This notebook reads the packaged saved outputs. It does not retrain the models; `SIT307_HD_Colab.ipynb` contains the full fresh-run workflow.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent
if not (ROOT / "results").exists() and (Path("/content") / "SIT307_11.1HD").exists():
    ROOT = Path("/content") / "SIT307_11.1HD"
if not (ROOT / "results/02_protocols/metrics.csv").exists():
    import runpy
    runpy.run_path(str(ROOT / "restore_saved_results.py"), run_name="__main__")
print("evidence root:", ROOT)


Saved results ready: 278 files checked; 278 files restored.
Next: python verify_results.py
evidence root: /workspace/scratch/36dda11e19b0/tmp/github_ready_qa/SIT307_11.1HD


In [2]:
data_path = ROOT / "data" / "heart.csv"
data = pd.read_csv(data_path)
key = data.drop(columns=["target"]).astype(str).agg("|".join, axis=1)
print("shape:", data.shape)
print("target counts:", data["target"].value_counts().sort_index().to_dict())
print("distinct predictor patterns:", key.nunique())
print("duplicate surplus rows:", len(data) - key.nunique())
print("sha256:", hashlib.sha256(data_path.read_bytes()).hexdigest())


shape: (1025, 14)
target counts: {0: 499, 1: 526}
distinct predictor patterns: 302
duplicate surplus rows: 723
sha256: ddb2996b2f4db2e00aad13f4518200179ff69f79093838e3c21ffa672ebec0f1


In [3]:
metrics = pd.read_csv(ROOT / "results/02_protocols/metrics.csv")
b = metrics[(metrics.seed < 30) & (metrics.arm == "B") & (metrics.model == "Stacking") & (metrics.weighting == "row")].sort_values("seed").reset_index(drop=True)
c = metrics[(metrics.seed < 30) & (metrics.arm == "C") & (metrics.model == "Stacking") & (metrics.weighting == "row")].sort_values("seed").reset_index(drop=True)
delta = (c.accuracy - b.accuracy) * 100
print(f"Protocol B accuracy: {b.accuracy.mean():.4f}")
print(f"Protocol C accuracy: {c.accuracy.mean():.4f}")
print(f"C minus B accuracy: {delta.mean():.2f} pp; wins/ties/losses = {(delta > 1e-9).sum()}/{(delta.abs() <= 1e-9).sum()}/{(delta < -1e-9).sum()}")
anchor = metrics[(metrics.seed == 42) & (metrics.arm == "C") & (metrics.model == "Stacking") & (metrics.weighting == "row")].iloc[0]
print("seed-42 C metrics:", {m: round(float(anchor[m]), 4) for m in ["accuracy", "precision", "recall", "f1", "roc_auc"]})


Protocol B accuracy: 0.7856
Protocol C accuracy: 0.8117
C minus B accuracy: 2.61 pp; wins/ties/losses = 21/1/8
seed-42 C metrics: {'accuracy': 0.8278, 'precision': 0.7767, 'recall': 0.8602, 'f1': 0.8163, 'roc_auc': 0.8617}


In [4]:
ablation = pd.read_csv(ROOT / "results/04_preprocessing/metrics.csv")
for arm, model in [("C_dedup", "Stacking"), ("C_onehot", "Stacking"), ("D", "Stacking"), ("D", "LR")]:
    sub = ablation[(ablation.seed < 30) & (ablation.arm == arm) & (ablation.model == model) & (ablation.weighting == "row")]
    print(arm, model, f"accuracy={sub.accuracy.mean():.4f}", f"AUC={sub.roc_auc.mean():.4f}")


C_dedup Stacking accuracy=0.8153 AUC=0.9007
C_onehot Stacking accuracy=0.8370 AUC=0.9038
D Stacking accuracy=0.8354 AUC=0.9050
D LR accuracy=0.8451 AUC=0.9127


In [5]:
nested = pd.read_csv(ROOT / "results/03_nested_validation/summary.csv").set_index("metric")
print(f"accuracy MAE: row CV={nested.loc['accuracy', 'row_cv_mae']*100:.2f} pp; grouped CV={nested.loc['accuracy', 'group_cv_mae']*100:.2f} pp")
print(f"grouped CV closer: accuracy={int(nested.loc['accuracy', 'group_closer'])}/10, F1={int(nested.loc['f1', 'group_closer'])}/10, AUC={int(nested.loc['roc_auc', 'group_closer'])}/10")


accuracy MAE: row CV=16.87 pp; grouped CV=4.98 pp
grouped CV closer: accuracy=9/10, F1=10/10, AUC=10/10


In [6]:
execution_files = sorted((ROOT / "verification" / "runs").glob("*/execution.json"))
execution = json.loads(execution_files[-1].read_text())
counts = {str(p.relative_to(ROOT)): len(list(p.glob("*.npz"))) for p in [ROOT / "results/02_protocols/runs", ROOT / "results/03_nested_validation/runs", ROOT / "results/04_preprocessing/runs", ROOT / "results/05_tuning/runs"]}
print("latest recorded execution:", execution["status"], "steps:", execution["steps_requested"])
print("saved NPZ counts:", counts)
print("generated number macros:", (ROOT / "paper/generated/numbers.tex").is_file())
print("verification command: python verify_results.py")


latest recorded execution: passed steps: [1, 2, 3, 4, 5, 6, 7, 8]
saved NPZ counts: {'results/02_protocols/runs': 93, 'results/03_nested_validation/runs': 20, 'results/04_preprocessing/runs': 93, 'results/05_tuning/runs': 31}
generated number macros: True
verification command: python verify_results.py
